# 🧠 Advanced Business Analytics, SQL Studio & Content Recommender
### *Strategy Formulation, SQL Business Queries & Machine Learning Recommendations*
---
**Objective**: Run business intelligence SQL queries on SQLite database, perform audience segmentation, and implement a content-based recommendation model.



In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Connect to Relational SQLite DB
db_path = Path('../data/processed/netflix.sqlite')
conn = sqlite3.connect(db_path)
print("[✓] Connected to SQLite database:", db_path.resolve())



## 1. Executive SQL Business Query: YoY Catalog Additions


In [ ]:
query_yoy = """
WITH yearly_data AS (
    SELECT 
        year_added,
        COUNT(*) AS total_added,
        SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) AS movies_added,
        SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) AS tv_shows_added
    FROM netflix_titles
    WHERE year_added >= 2012
    GROUP BY year_added
)
SELECT 
    year_added,
    total_added,
    movies_added,
    tv_shows_added,
    LAG(total_added, 1) OVER (ORDER BY year_added) AS prev_year_total,
    ROUND((total_added - LAG(total_added, 1) OVER (ORDER BY year_added)) * 100.0 / 
          NULLIF(LAG(total_added, 1) OVER (ORDER BY year_added), 0), 2) AS yoy_growth_pct
FROM yearly_data
ORDER BY year_added ASC;
"""

pd.read_sql_query(query_yoy, conn)



## 2. SQL Query: Director-Actor Collaboration Power Duos


In [ ]:
query_duos = """
SELECT 
    td.director AS Director,
    tc.actor AS Actor,
    COUNT(DISTINCT td.show_id) AS Collaborative_Titles
FROM title_directors td
JOIN title_cast tc ON td.show_id = tc.show_id
WHERE td.director != 'Unknown Director' AND tc.actor != 'Unknown Cast'
GROUP BY td.director, tc.actor
HAVING Collaborative_Titles >= 3
ORDER BY Collaborative_Titles DESC
LIMIT 12;
"""

pd.read_sql_query(query_duos, conn)



## 3. SQL Query: International vs US Content Expansion


In [ ]:
query_intl = """
SELECT 
    year_added,
    COUNT(*) AS total_additions,
    SUM(CASE WHEN country LIKE '%United States%' THEN 1 ELSE 0 END) AS us_titles,
    SUM(CASE WHEN country NOT LIKE '%United States%' AND country != 'Unknown Country' THEN 1 ELSE 0 END) AS international_titles,
    ROUND(
        SUM(CASE WHEN country NOT LIKE '%United States%' AND country != 'Unknown Country' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        1
    ) AS international_pct
FROM netflix_titles
WHERE year_added >= 2014
GROUP BY year_added
ORDER BY year_added ASC;
"""

pd.read_sql_query(query_intl, conn)



## 4. Machine Learning Content Recommendation Engine
We construct a TF-IDF vectorizer over the metadata soup (Title, Genre, Cast, Director, Country, Age Group, Description) and compute Cosine Similarity.



In [ ]:
df = pd.read_csv('../data/processed/netflix_cleaned.csv')

def create_soup(row):
    title = str(row.get('title', ''))
    genres = str(row.get('listed_in', '')).replace(',', ' ')
    director = str(row.get('director_clean', '')).replace(',', ' ')
    cast = ' '.join(str(row.get('cast_clean', '')).split(',')[:4])
    country = str(row.get('country_clean', '')).replace(',', ' ')
    rating = str(row.get('rating_clean', ''))
    desc = str(row.get('description', ''))
    audience = str(row.get('target_audience', ''))
    return f"{title} {genres} {genres} {director} {cast} {country} {rating} {audience} {desc}".lower()

df['metadata_soup'] = df.apply(create_soup, axis=1)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=12000)
tfidf_matrix = tfidf.fit_transform(df['metadata_soup'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

# Recommender function
def get_recommendations(title, top_n=5):
    title_matches = df[df['title'].str.lower() == title.lower()]
    if title_matches.empty:
        title_matches = df[df['title'].str.lower().str.contains(title.lower())]
        if title_matches.empty:
            return f"No title matching '{title}' found."
    
    idx = title_matches.index[0]
    query_vec = tfidf_matrix[idx]
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    sorted_idx = sim_scores.argsort()[::-1]
    
    results = []
    for i in sorted_idx:
        if i == idx:
            continue
        row = df.iloc[i]
        results.append({
            'Title': row['title'],
            'Type': row['type'],
            'Similarity': f"{round(sim_scores[i] * 100, 1)}%",
            'Release_Year': row['release_year'],
            'Genres': row['listed_in'],
            'Director': row['director_clean']
        })
        if len(results) >= top_n:
            break
            
    return pd.DataFrame(results)

# Test Recommendation for 'Stranger Things'
get_recommendations('Stranger Things', top_n=5)



## 5. Strategic Recommendations for C-Suite
1. **Accelerate TV Series Original Franchises**: TV Shows drive platform retention and reduce subscriber churn, yet represent only ~30% of catalog.
2. **Local-for-Global Content Strategy**: International hubs (India, South Korea, Spain, UK) show superior growth velocity and cross-border viewership.
3. **Optimize Runtime Clustering**: The consumer sweet spot for feature films is 90-105 minutes; avoid excessive 150+ min catalog padding without marquee IP.
4. **Seasonal Release Coordination**: Align high-budget tentpole drops with July and December holiday peaks to capture maximum streaming hours.

